# DeltaGrad results analysis

Loads the `results/<task>/<optimizer>_results.pkl` files written by
`experiments/run_task.py` and works through the claims
`deltagradpaperplan.pdf` actually makes:

| Section | Question it answers | Paper claim under test |
|---|---|---|
| 3 Leaderboard | Which optimizer wins, by how much? | Table 1 primary metrics |
| 4 Learning curves | How fast, how smoothly does each converge? | Sec. 1 "improving generalization" |
| 5 Gradient variance | Does DeltaGrad cut variance? | Sec. 4.1 "order-of-magnitude reduction relative to Adam" |
| 6 The $R_t$ mechanism | Does $R_t$ actually track gradient disagreement? | Sec. 2/3 -- $R_t$-variance correlation |
| 7 $R_t$ transforms | Which slice of each transform's curve does training use? | Sec. 3.2 -- the six $R_t$ transform options |
| 8 Seed stability | Is DeltaGrad less seed-sensitive? | Sec. 4.1 "accuracy standard deviation" |
| 9 Significance | Is the gap real or seed noise? | (not in the plan -- but needed to claim any of it) |
| 10 Wall-clock overhead | What does the extra state cost? | Sec. 4.2 "overhead remains below 0.5%" |
| 11 Noise memorization | Does DeltaGrad resist label noise? | Sec. 4.1 "noise memorization vs. generalization" |
| 12 Cross-task ranks | Does the win generalize across tasks? | Table 1 as a whole |

Unlike `deltagrad/viz.py` (which renders the final paper figures at
paper-column size, and compares exactly two optimizers), everything here
handles **any number of optimizers** and is sized for reading on screen.

**Prerequisite:** at least one `results/<task>/<optimizer>_results.pkl`. Generate
them with, e.g.

```bash
python -m experiments.run_task --task cifar100_noise_20 --optimizer windowed
python -m experiments.run_task --task cifar100_noise_20 --optimizer adam
```

Every section degrades gracefully -- with only one optimizer's results present
the comparison sections say what is missing instead of failing.

## 1. Setup

Finds the repo (locally: walks up from the working directory; on Colab: pulls or
clones it), then imports the toolkit. No GPU needed -- this notebook only reads
`.pkl` files.

In [ ]:
import os
import subprocess
import sys


def _find_repo_root(start=None):
    """Nearest ancestor directory containing experiments/configs.py."""
    path = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(path, "experiments", "configs.py")):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            return None
        path = parent


REPO_ROOT = _find_repo_root()
IN_COLAB = "google.colab" in sys.modules

if REPO_ROOT is None and IN_COLAB:
    from google.colab import userdata

    GITHUB_REPO = "xandasoneill/deltagrad_optimizer"
    REPO_ROOT = "/content/deltagrad_optimizer"
    # Passed as an argv list rather than an IPython `!` line so the token never
    # gets echoed into this notebook's saved output.
    remote = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{GITHUB_REPO}.git"
    if os.path.isdir(os.path.join(REPO_ROOT, ".git")):
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "origin", "master"], check=False)
    else:
        subprocess.run(["git", "clone", remote, REPO_ROOT], check=True)

if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repo -- run this notebook from inside "
                       "the DeltaGrad checkout, or set REPO_ROOT by hand.")

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

### 1.1 Toolkit -- loading and aggregation

Each results `.pkl` holds, per benchmark run:

| Key | Shape | Meaning |
|---|---|---|
| `acc_history` | (runs, epochs) | test accuracy % -- or **reconstruction loss** on VAE tasks |
| `loss_history` | (runs, epochs) | mean training loss |
| `variance_history` | (runs, measurements) | gradient variance, sampled every 10th batch |
| `r_history` | (runs, measurements) | mean $R_t$ -- **empty for baselines**, which have no $R_t$ |
| `all_timestamps` | (runs, epochs) | *cumulative* net compute time at each epoch's end |
| `optimizer_hyperparameters`, `seeds`, `batch_size`, ... | | provenance |

In [ ]:
import glob
import os

import joblib
import numpy as np
import pandas as pd
import torch
from scipy import stats

from deltagrad.optimizers import R_TRANSFORMS
from experiments.configs import TASK_REGISTRY, OPTIMIZER_KEYS

RESULTS_ROOT = "results"

# DeltaGrad variants get solid lines, baselines dashed, so the two families stay
# distinguishable at a glance even when 7 curves share an axis.
OPT_STYLE = {
    "windowed":     ("#008080", "-"),
    "ema":          ("#7b3fa0", "-"),
    "adam":         ("#ff7f0e", "--"),
    "adamw":        ("#d62728", "--"),
    "sgd_momentum": ("#2ca02c", "--"),
    "adagrad":      ("#8c564b", "--"),
    "rmsprop":      ("#17becf", "--"),
}
DELTAGRAD_KEYS = ("windowed", "ema")

# experiments/sweep_r_transforms.py saves each of DeltaGradEMA's Sec. 3.2
# transforms as its own "optimizer" (ema_exp, ema_sigmoid, ...) so they compete
# in every comparison here; give each its own colour rather than 6 identical greys.
TRANSFORM_COLORS = {
    "linear":  "#4c72b0",
    "exp":     "#7b3fa0",
    "inverse": "#dd8452",
    "power":   "#55a868",
    "sigmoid": "#c44e52",
    "zscore":  "#937860",
}


def style(optimizer):
    if optimizer.startswith("ema_"):
        return TRANSFORM_COLORS.get(optimizer[len("ema_"):], "#7f7f7f"), "-"
    return OPT_STYLE.get(optimizer, ("#7f7f7f", ":"))


def discover(root=RESULTS_ROOT):
    """Every results/<task>/<optimizer>_results.pkl, as a DataFrame.

    Skips results/legacy/ (pre-reorg, different schema) and results/_smoke_runs/
    (throwaway 1-epoch wiring checks) -- both live a directory deeper than the
    glob reaches, but the leading-underscore guard keeps that explicit."""
    rows = []
    for path in sorted(glob.glob(os.path.join(root, "*", "*_results.pkl"))):
        task = os.path.basename(os.path.dirname(path))
        if task.startswith("_") or task == "legacy":
            continue
        payload = joblib.load(path)
        rows.append({
            "task": task,
            "optimizer": os.path.basename(path)[: -len("_results.pkl")],
            "runs": payload.get("number_runs"),
            "epochs": payload.get("epochs"),
            "batch_size": payload.get("batch_size"),
            "model": payload.get("model_name"),
            "device": payload.get("device"),
            "hyperparameters": payload.get("optimizer_hyperparameters"),
            "path": path,
        })
    return pd.DataFrame(rows)


_CACHE = {}


def load(task, optimizer, root=RESULTS_ROOT):
    key = (root, task, optimizer)
    if key not in _CACHE:
        _CACHE[key] = joblib.load(os.path.join(root, task, f"{optimizer}_results.pkl"))
    return _CACHE[key]


def ordered_optimizers(names):
    """OPTIMIZER_KEYS order first, then anything else alphabetically.

    The tail matters: experiments/sweep_r_transforms.py writes `ema_exp`,
    `ema_sigmoid` and friends, which are perfectly good competitors but are not
    in OPTIMIZER_KEYS -- filtering to that list instead of ordering by it would
    silently drop every transform from the comparison."""
    names = list(names)
    return ([o for o in OPTIMIZER_KEYS if o in names]
            + sorted(o for o in names if o not in OPTIMIZER_KEYS))


def load_task(task, root=RESULTS_ROOT):
    """{optimizer: results} for `task`, in ordered_optimizers order."""
    found = discover(root).query("task == @task")["optimizer"].tolist()
    return {o: load(task, o, root) for o in ordered_optimizers(found)}


def lower_is_better(task):
    """VAE tasks score reconstruction loss; everything else scores accuracy."""
    config = TASK_REGISTRY.get(task)
    return config is not None and config.task_type == "vae"


def metric_name(task):
    return "Test recon. loss" if lower_is_better(task) else "Test accuracy (%)"


def as_matrix(results, key):
    """(runs, points) float array. Runs are truncated to the shortest one so a
    partially-finished run can still be plotted; missing keys give (0, 0)."""
    sequences = [np.asarray(s, dtype=float) for s in (results.get(key) or [])]
    sequences = [s for s in sequences if s.size]
    if not sequences:
        return np.empty((0, 0))
    width = min(len(s) for s in sequences)
    return np.stack([s[:width] for s in sequences])


def final_metric(results):
    """Final-epoch metric per run -- what run_task.py reports and what tuning targets."""
    matrix = as_matrix(results, "acc_history")
    return matrix[:, -1] if matrix.size else np.empty(0)


def best_metric(results, task):
    matrix = as_matrix(results, "acc_history")
    return matrix.min(axis=1) if lower_is_better(task) else matrix.max(axis=1)


def peak_to_final_gap(results, task):
    """Peak-minus-final metric per run: how much of its own best result the
    optimizer gave back by the end. Positive = degraded after peaking, the
    signature of memorizing noisy labels."""
    matrix = as_matrix(results, "acc_history")
    if lower_is_better(task):
        return matrix[:, -1] - matrix.min(axis=1)
    return matrix.max(axis=1) - matrix[:, -1]


def seconds_per_epoch(results):
    """(runs, epochs) per-epoch durations, differenced out of the cumulative stamps."""
    cumulative = as_matrix(results, "all_timestamps")
    if cumulative.size == 0:
        return np.empty((0, 0))
    return np.diff(cumulative, axis=1, prepend=0.0)


def epochs_to_target(results, target, task):
    """First epoch (1-indexed) reaching `target`; NaN for runs that never do."""
    matrix = as_matrix(results, "acc_history")
    hits = matrix <= target if lower_is_better(task) else matrix >= target
    reached = hits.any(axis=1)
    return np.where(reached, hits.argmax(axis=1) + 1.0, np.nan)


def smooth(values, window):
    """Centred rolling mean -- gradient variance is far too spiky to read raw."""
    if window <= 1 or len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")


def binned_median(x, y, bins=25, min_count=5):
    """Median `y` per `x` bin -- a trend line for the R-vs-variance scatter.

    Preferred over a LOWESS fit here for two reasons: it needs no statsmodels,
    and it cannot invent values outside the data, whereas a linear-space LOWESS
    plotted on a log axis dives to zero wherever the right tail thins out."""
    edges = np.linspace(x.min(), x.max(), bins + 1)
    slot = np.digitize(x, edges)
    centers, medians = [], []
    for index in range(1, len(edges)):
        members = y[slot == index]
        if len(members) >= min_count:
            centers.append(0.5 * (edges[index - 1] + edges[index]))
            medians.append(np.median(members))
    return np.array(centers), np.array(medians)


def reference_optimizer(runs_by_optimizer):
    """The optimizer everything else is measured against. Adam is the plan's
    stated comparison point; failing that, any other baseline.

    None when only one optimizer has been run: "1.00x less variance than
    itself" is noise, so callers drop the relative columns entirely rather
    than printing a tautology."""
    if len(runs_by_optimizer) < 2:
        return None
    for candidate in ("adam", "adamw", "sgd_momentum", "rmsprop", "adagrad"):
        if candidate in runs_by_optimizer:
            return candidate
    return next(iter(runs_by_optimizer))


print(f"Toolkit loaded. Known optimizers: {', '.join(OPTIMIZER_KEYS)}")

### 1.2 Toolkit -- plot style and saving

`SAVE_FIGURES = True` mirrors every figure to `results/figures/<task>/` as both
PNG (for the repo/README) and PDF (vector, for LaTeX).

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--",
})

SAVE_FIGURES = True
FIG_ROOT = "results/figures"


def save_fig(fig, name, subdir):
    if not SAVE_FIGURES:
        return
    out_dir = os.path.join(FIG_ROOT, subdir)
    os.makedirs(out_dir, exist_ok=True)
    for extension in ("png", "pdf"):
        fig.savefig(os.path.join(out_dir, f"{name}.{extension}"), bbox_inches="tight")
    print(f"saved -> {os.path.join(out_dir, name)}.png/.pdf")


def _note(message):
    """Explains why a section produced nothing, instead of raising."""
    print(f"[skipped] {message}")

## 2. What results exist?

Everything below reads from this inventory, so start here to see which
task/optimizer pairs have actually been run.

In [ ]:
inventory = discover()

if inventory.empty:
    _note("No results yet. Run e.g. "
          "`python -m experiments.run_task --task cifar100_noise_20 --optimizer adam`")
else:
    display(inventory.drop(columns=["path"]))
    coverage = (inventory.assign(done=1)
                .pivot_table(index="task", columns="optimizer", values="done", fill_value=0)
                .astype(int))
    print("\nCoverage (1 = results present):")
    display(coverage)
    missing = {task: [o for o in OPTIMIZER_KEYS if o not in group["optimizer"].values]
               for task, group in inventory.groupby("task")}
    for task, gaps in missing.items():
        if gaps:
            print(f"{task}: not yet run -> {', '.join(gaps)}")

### 2.1 Pick the task to analyse

Defaults to whichever task has the most optimizers run against it (the most
informative one to look at). Override `TASK` by hand to analyse a different one.

In [ ]:
if inventory.empty:
    raise SystemExit("Nothing to analyse yet -- run a benchmark first.")

TASK = inventory.groupby("task").size().idxmax()   # <- override manually if you like

runs = load_task(TASK)
REFERENCE = reference_optimizer(runs)
LOWER_BETTER = lower_is_better(TASK)

config = TASK_REGISTRY.get(TASK)
print(f"Task       : {TASK}")
print(f"Model      : {config.model_cls.__name__ if config else '?'}")
print(f"Metric     : {metric_name(TASK)}  ({'lower' if LOWER_BETTER else 'higher'} is better)")
print(f"Optimizers : {', '.join(runs)}")
print(f"Reference  : {REFERENCE or '(none -- only one optimizer run, so nothing to compare against)'}")
for optimizer, result in runs.items():
    print(f"  {optimizer:<14} {result['number_runs']} runs x {result['epochs']} epochs "
          f"| batch {result['batch_size']} | {result['device']} | {result['optimizer_hyperparameters']}")

## 3. Leaderboard

One row per optimizer. The columns worth arguing about:

- **final** -- mean $\pm$ std of the last epoch's metric across seeds. This is what
  `run_task.py` prints and what `tune_hyperparams.py` optimises, so it is the
  headline number.
- **best** -- mean of each run's *own* best epoch. `best` far above `final` means
  the optimizer peaked early and then decayed.
- **peak-final gap** -- exactly that decay, per run. Under label noise this is the
  memorization signature: the model fits the corrupted labels and test accuracy
  slides back off its peak.
- **epochs to target** -- speed. Target is 95% of the best final score any
  optimizer reached on this task (105% for loss), so it is comparable across the row.
- **grad var** -- mean gradient variance over training, and the reduction factor
  against the reference optimizer. The plan claims an order of magnitude (10x).
- **s/epoch** -- wall clock, and the overhead against the reference. The plan
  budgets under 0.5%.

In [ ]:
def leaderboard(task, runs_by_optimizer, reference=None):
    reference = reference or reference_optimizer(runs_by_optimizer)
    lower = lower_is_better(task)

    finals = {o: final_metric(r) for o, r in runs_by_optimizer.items()}
    best_final = (min if lower else max)(v.mean() for v in finals.values())
    # 95% of the best score for accuracy; within 105% of it for loss.
    target = best_final * (1.05 if lower else 0.95)

    rows = []
    for optimizer, result in runs_by_optimizer.items():
        final = finals[optimizer]
        variance = as_matrix(result, "variance_history")
        r_values = as_matrix(result, "r_history")
        per_epoch = seconds_per_epoch(result)
        # All-NaN when no run ever hit the target; nanmean would warn on that.
        reached = epochs_to_target(result, target, task)
        rows.append({
            "optimizer": optimizer,
            "runs": len(final),
            "final": final.mean(),
            "final_std": final.std(ddof=1) if len(final) > 1 else np.nan,
            "best": best_metric(result, task).mean(),
            "peak_final_gap": peak_to_final_gap(result, task).mean(),
            "epochs_to_target": np.nan if np.isnan(reached).all() else np.nanmean(reached),
            "grad_var": variance.mean() if variance.size else np.nan,
            "mean_R": r_values.mean() if r_values.size else np.nan,
            "s_per_epoch": per_epoch.mean() if per_epoch.size else np.nan,
        })

    table = pd.DataFrame(rows).set_index("optimizer")
    if reference in table.index:
        table["var_reduction_x"] = table.loc[reference, "grad_var"] / table["grad_var"]
        table["overhead_%"] = (table["s_per_epoch"] / table.loc[reference, "s_per_epoch"] - 1) * 100
    return table.sort_values("final", ascending=lower), target


board, TARGET = leaderboard(TASK, runs, REFERENCE)

print(f"{TASK} | target for 'epochs_to_target': {TARGET:.2f} "
      f"({'<=' if LOWER_BETTER else '>='} this counts as converged)")
if REFERENCE:
    print(f"'var_reduction_x' and 'overhead_%' are relative to {REFERENCE}; "
          f"reduction > 1 means less gradient variance.\n")
else:
    print("Relative columns omitted -- run a second optimizer on this task "
          "to get variance-reduction and overhead comparisons.\n")
display(board.style.format({
    "final": "{:.3f}", "final_std": "{:.3f}", "best": "{:.3f}",
    "peak_final_gap": "{:+.3f}", "epochs_to_target": "{:.1f}",
    "grad_var": "{:.3e}", "mean_R": "{:.3f}", "s_per_epoch": "{:.2f}",
    "var_reduction_x": "{:.2f}x", "overhead_%": "{:+.2f}%",
}, na_rep="--"))

## 4. Learning curves

Mean across seeds, with a $\pm 1$ std band. The band is the interesting part: two
optimizers can share a mean curve while one is far more seed-dependent, and a
wide band means the headline number in section 3 is partly luck.

Left is the metric being reported (test accuracy, or reconstruction loss for
VAE); right is training loss, which shows convergence speed and -- when it keeps
falling while test accuracy stalls -- memorization.

In [ ]:
def plot_learning_curves(task, runs_by_optimizer):
    if not runs_by_optimizer:
        return _note("No results loaded.")

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
    for optimizer, result in runs_by_optimizer.items():
        color, dashes = style(optimizer)
        for ax, key in zip(axes, ("acc_history", "loss_history")):
            matrix = as_matrix(result, key)
            if matrix.size == 0:
                continue
            mean, std = matrix.mean(axis=0), matrix.std(axis=0, ddof=0)
            epochs = np.arange(1, matrix.shape[1] + 1)
            ax.plot(epochs, mean, color=color, linestyle=dashes, linewidth=1.8, label=optimizer)
            ax.fill_between(epochs, mean - std, mean + std, color=color, alpha=0.15, linewidth=0)

    direction = "lower is better" if lower_is_better(task) else "higher is better"
    axes[0].set(xlabel="Epoch", ylabel=metric_name(task),
                title=f"{task} -- {metric_name(task)} ({direction})")
    axes[1].set(xlabel="Epoch", ylabel="Training loss", title=f"{task} -- training loss")
    axes[1].set_yscale("log")
    axes[0].legend(fontsize=9, ncol=2)
    fig.suptitle("Mean across seeds, shaded $\\pm$1 std", fontsize=10, y=1.04)
    save_fig(fig, "learning_curves", task)
    plt.show()


plot_learning_curves(TASK, runs)

## 5. Gradient variance

The plan's central mechanical claim (Sec. 4.1): DeltaGrad should show an
**order-of-magnitude reduction** in global mean gradient variance relative to
Adam.

Variance is measured every 10th batch by `deltagrad/training.py`, by re-sampling
the batch 8 times and taking the variance of the resulting gradients -- so it is
an estimate of how noisy the gradient signal itself is, not of the update size.
Raw traces are extremely spiky, hence the rolling mean; the log axis is
essential because the whole point is orders of magnitude.

In [ ]:
def plot_gradient_variance(task, runs_by_optimizer, reference=None, window=15):
    reference = reference or reference_optimizer(runs_by_optimizer)
    usable = {o: r for o, r in runs_by_optimizer.items()
              if as_matrix(r, "variance_history").size}
    if not usable:
        return _note("No gradient-variance measurements in these results.")

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4),
                             gridspec_kw={"width_ratios": [2, 1]})
    means = {}
    for optimizer, result in usable.items():
        color, dashes = style(optimizer)
        matrix = np.clip(as_matrix(result, "variance_history"), 1e-12, None)
        means[optimizer] = matrix.mean()
        trace = smooth(matrix.mean(axis=0), window)
        axes[0].plot(np.arange(len(trace)), trace, color=color, linestyle=dashes,
                     linewidth=1.6, label=f"{optimizer} (mean {matrix.mean():.2e})")
        # Individual seeds, faint, to show the spread the mean hides.
        for run in matrix:
            single = smooth(run, window)
            axes[0].plot(np.arange(len(single)), single, color=color, alpha=0.12, linewidth=0.6)

    axes[0].set(xlabel=f"Measurement index (every 10th batch, rolling mean w={window})",
                ylabel="Gradient variance", yscale="log",
                title=f"{task} -- gradient variance over training")
    axes[0].legend(fontsize=8)

    order = sorted(means, key=means.get)
    axes[1].barh(order, [means[o] for o in order],
                 color=[style(o)[0] for o in order], alpha=0.85)
    axes[1].set(xscale="log", xlabel="Mean gradient variance (log)",
                title="Global mean")
    if reference in means:
        axes[1].axvline(means[reference], color="black", linestyle=":", linewidth=1)
        for y, optimizer in enumerate(order):
            factor = means[reference] / means[optimizer]
            axes[1].text(means[optimizer], y, f"  {factor:.1f}x", va="center", fontsize=9)
    save_fig(fig, "gradient_variance", task)
    plt.show()

    if reference in means:
        print(f"Reduction factors vs {reference} (>1 = less variance, "
              f"the plan's claim is ~10x for DeltaGrad):")
        for optimizer in order:
            print(f"  {optimizer:<14} {means[reference] / means[optimizer]:8.2f}x"
                  f"   (mean {means[optimizer]:.3e})")
    else:
        print("Absolute means only -- run a baseline (e.g. adam) on this task "
              "to get the reduction factor the plan's claim is stated in.")
        for optimizer in order:
            print(f"  {optimizer:<14} mean {means[optimizer]:.3e}")


plot_gradient_variance(TASK, runs, REFERENCE)

## 6. The $R_t$ mechanism

$R_t$ is the reliability metric both DeltaGrad variants scale their update by
($\theta_{t+1} = \theta_t - \mu (R_t \odot m_t)$). It is only recorded for
`windowed` and `ema` -- baselines have no such state, so they are absent here by
construction, not by omission.

$R_t \in [A, B]$ (default $[0.1, 1.0]$): near $B$ the recent gradients agree and the
full step is taken; near $A$ they disagree and the step is damped.

### 6.1 Does $R_t$ move during training?

A flat $R_t$ pinned at either clamp bound means the mechanism is inert -- the
optimizer has degenerated into plain momentum SGD (at $B$) or a uniformly
shrunken version of it (at $A$). What you want to see is $R_t$ dropping when the
overlaid gradient variance rises.

In [ ]:
def plot_r_trajectory(task, runs_by_optimizer, window=15):
    usable = {o: r for o, r in runs_by_optimizer.items() if as_matrix(r, "r_history").size}
    if not usable:
        return _note("No R values recorded -- only DeltaGrad (windowed/ema) tracks R.")

    fig, axes = plt.subplots(1, len(usable), figsize=(6.2 * len(usable), 4.2), squeeze=False)
    for ax, (optimizer, result) in zip(axes[0], usable.items()):
        color, _ = style(optimizer)
        r_matrix = as_matrix(result, "r_history")
        mean, std = r_matrix.mean(axis=0), r_matrix.std(axis=0, ddof=0)
        x = np.arange(len(mean))
        ax.plot(x, mean, color=color, linewidth=1.6, label="$R_t$ (mean)")
        ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)
        ax.set(xlabel="Measurement index", ylabel="$R_t$",
               title=f"{optimizer}: $R_t$ range [{r_matrix.min():.3f}, {r_matrix.max():.3f}]")

        variance = as_matrix(result, "variance_history")
        if variance.size:
            twin = ax.twinx()
            width = min(variance.shape[1], r_matrix.shape[1])
            trace = smooth(variance.mean(axis=0)[:width], window)
            twin.plot(np.arange(len(trace)), trace,
                      color="0.35", linewidth=1.0, linestyle=":", label="grad variance")
            twin.set(ylabel="Gradient variance", yscale="log")
            twin.grid(False)
        ax.legend(loc="lower left", fontsize=9)
    save_fig(fig, "r_trajectory", task)
    plt.show()


plot_r_trajectory(TASK, runs)

### 6.2 Is $R_t$ anti-correlated with gradient variance?

This is the claim the whole design rests on: $R_t$ is supposed to be *low exactly
when the gradient is noisy*. Both series are recorded at the same measurement
points, so they align index-for-index.

A **negative** Pearson/Spearman $r$ means the damping fires when it should. A
correlation near zero means $R_t$ is responding to something other than the noise
it is meant to track. Spearman is reported alongside Pearson because the
relationship need not be linear -- and because gradient variance is heavy-tailed
enough that a handful of spikes can dominate Pearson on their own.

In [ ]:
def plot_r_vs_variance(task, runs_by_optimizer):
    usable = {o: r for o, r in runs_by_optimizer.items()
              if as_matrix(r, "r_history").size and as_matrix(r, "variance_history").size}
    if not usable:
        return _note("Needs both R and variance histories -- DeltaGrad runs only.")

    fig, axes = plt.subplots(1, len(usable), figsize=(6.0 * len(usable), 4.4), squeeze=False)
    summary = []
    for ax, (optimizer, result) in zip(axes[0], usable.items()):
        color, _ = style(optimizer)
        r_matrix, v_matrix = as_matrix(result, "r_history"), as_matrix(result, "variance_history")
        width = min(r_matrix.shape[1], v_matrix.shape[1])
        r_flat = r_matrix[:, :width].ravel()
        v_flat = np.clip(v_matrix[:, :width].ravel(), 1e-12, None)

        for run_index in range(r_matrix.shape[0]):
            run_r, run_v = r_matrix[run_index, :width], v_matrix[run_index, :width]
            if np.std(run_r) == 0 or np.std(run_v) == 0:
                continue   # a run pinned at a clamp bound has no correlation to measure
            summary.append({
                "optimizer": optimizer, "run": run_index + 1,
                "pearson_r": stats.pearsonr(run_r, run_v)[0],
                "spearman_r": stats.spearmanr(run_r, run_v)[0],
            })

        ax.scatter(r_flat, v_flat, s=4, alpha=0.18, color=color, edgecolors="none")
        centers, medians = binned_median(r_flat, v_flat)
        if len(centers):
            ax.plot(centers, medians, color="black", linewidth=1.8, linestyle="--",
                    label="binned median")
            ax.legend(fontsize=8, loc="lower left")
        pearson_r, pearson_p = stats.pearsonr(r_flat, v_flat)
        spearman_r, _ = stats.spearmanr(r_flat, v_flat)
        ax.set(xlabel="$R_t$", ylabel="Gradient variance", yscale="log",
               title=(f"{optimizer}: pooled Pearson r={pearson_r:.3f} "
                      f"(p={pearson_p:.1e})\nSpearman $\\rho$={spearman_r:.3f}"))

    save_fig(fig, "r_vs_variance", task)
    plt.show()

    if summary:
        per_run = pd.DataFrame(summary)
        print("Per-run correlations (negative = R drops when the gradient gets noisy):")
        display(per_run.groupby("optimizer")[["pearson_r", "spearman_r"]]
                .agg(["mean", "std", "min", "max"]).round(3))


plot_r_vs_variance(TASK, runs)

## 7. DeltaGrad-EMA's $R_t$ transforms

Section 6 asked whether $R_t$ tracks gradient disagreement. This one asks how it
is *produced*: `DeltaGradEMA` maps its accumulated disagreement $\hat{S}_t$ to
$R_t$ through one of the six candidate transforms in Sec. 3.2 of the plan, then
clamps to $[A, B]$.

$\hat{S}_t$ is bounded in $[0, 1]$ by construction ($\phi_t$ is a normalised
absolute difference, and $S_t$ is an EMA of it), so the interesting question for
each transform is **which slice of $[0, 1]$ training actually visits** -- a
transform whose whole interesting behaviour sits at $\hat{S} > 0.5$ is inert if
training never leaves $\hat{S} < 0.1$.

### 7.1 The six curves

Drawn by calling `deltagrad.optimizers.R_TRANSFORMS` directly, so these are
literally the functions the optimizer applies, at the class-default shape
parameters. The shaded band is the default clamp range $[A, B] = [0.1, 1.0]$;
anything outside it gets flattened onto the boundary.

`zscore` is absent: it standardises $\hat{S}$ against running per-parameter
$\mu_S$/$\sigma_S$ state, so it has no fixed curve to draw -- it is a different
mapping at every step. That self-calibration is the point of Option 5, and it is
why the section below plots it from samples only.

In [ ]:
def transform_curve(spec, grid):
    """R(S_hat) over `grid`, evaluated by deltagrad's own transform functions so
    a drawn curve can never drift from the applied one. None for zscore, whose
    mapping depends on running state rather than S_hat alone."""
    name = spec["r_transform"]
    if name == "zscore":
        return None
    _, function = R_TRANSFORMS[name]
    values = function(torch.as_tensor(grid, dtype=torch.float32),
                      gamma=spec.get("gamma", 1.0), power_p=spec.get("power_p", 0.5),
                      tau=spec.get("tau", 0.0), s=spec.get("s", 1.0),
                      mu_S_hat=None, sigma_S_hat=None,
                      zscore_k=spec.get("zscore_k", 2.0), epsilon=1e-8)
    return values.numpy()


DEFAULT_SPEC = {"r_transform": None, "gamma": 1.0, "power_p": 0.5, "tau": 0.0,
                "s": 1.0, "zscore_k": 2.0, "A": 0.1, "B": 1.0}


def plot_transform_atlas():
    grid = np.linspace(0.0, 1.0, 400)
    fig, ax = plt.subplots(figsize=(8.0, 5.0))
    ax.axhspan(DEFAULT_SPEC["A"], DEFAULT_SPEC["B"], color="0.85", alpha=0.5, zorder=0,
               label=f"clamp [{DEFAULT_SPEC['A']}, {DEFAULT_SPEC['B']}]")
    for name, (option, _) in R_TRANSFORMS.items():
        curve = transform_curve({**DEFAULT_SPEC, "r_transform": name}, grid)
        if curve is None:
            continue
        ax.plot(grid, curve, color=TRANSFORM_COLORS[name], linewidth=2.0,
                label=f"{name} (Option {option})")
    ax.set(xlabel="$\\hat{S}_t$  (accumulated gradient disagreement)", ylabel="$R_t$",
           title="Sec. 3.2 R-transforms at class-default shape parameters",
           xlim=(0, 1))
    ax.legend(fontsize=9)
    save_fig(fig, "transform_atlas", "_transforms")
    plt.show()

    print("Range of R each transform spans over S_hat in [0, 1], before clamping:")
    for name in R_TRANSFORMS:
        curve = transform_curve({**DEFAULT_SPEC, "r_transform": name}, grid)
        if curve is None:
            print(f"  {name:<9} state-dependent (no fixed curve)")
        else:
            print(f"  {name:<9} R in [{curve.min():.3f}, {curve.max():.3f}]"
                  f"   span {curve.max() - curve.min():.3f}")


plot_transform_atlas()

### 7.2 Where training actually lands on the curve

`DeltaGradEMA(sample_every=N)` records an evenly-strided subsample of the live
$(\hat{S}_t, R_t)$ pairs every $N$ optimizer steps. The striding is deliberate:
a random subsample would draw from the global RNG and desynchronise dropout and
shuffling, so an instrumented run would no longer be the run you meant to
measure. (`tests/test_ema_optimizer.py` asserts an instrumented run lands on
byte-identical parameters.)

To generate the data:

```bash
python -m experiments.sweep_r_transforms --task mnist_logreg --epochs 15
```

Each panel below draws the transform's analytic curve, shades the clamp band,
and scatters the sampled operating points coloured by training step, so drift
over training is visible. Points sitting *off* the curve are exactly the ones
the clamp caught. The rug along the top shows the marginal $\hat{S}$ density --
the "range each transform is working with".

In [ ]:
def transform_sample_frame(results):
    """results['transform_samples'] (per run -> per capture) as one tidy frame."""
    pieces = []
    for run_index, captures in enumerate(results.get("transform_samples") or []):
        for capture in captures:
            pieces.append(pd.DataFrame({
                "run": run_index,
                "step": capture["step"],
                "S_hat": capture["S_hat"],
                "R": capture["R"],
                "param_index": capture["param_index"],
            }))
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()


def transform_runs(task, root=RESULTS_ROOT):
    """{label: (spec, frame)} for every optimizer on `task` that recorded samples."""
    found = {}
    for optimizer, result in load_task(task, root).items():
        frame = transform_sample_frame(result)
        if frame.empty:
            continue
        spec = result.get("transform_spec") or {
            **DEFAULT_SPEC, "r_transform": optimizer.replace("ema_", "")}
        found[optimizer] = (spec, frame)
    return found


def plot_transform_operating_points(task, root=RESULTS_ROOT, max_points=4000):
    sampled = transform_runs(task, root)
    if not sampled:
        return _note(
            f"No (S_hat, R) samples for '{task}'. Generate them with "
            f"`python -m experiments.sweep_r_transforms --task {task}` (all 6 transforms), "
            f"or `python -m experiments.run_task --task {task} --optimizer ema "
            f"--sample-transform-every 50` (whichever transform the config uses).")

    columns = min(3, len(sampled))
    rows = -(-len(sampled) // columns)
    fig, axes = plt.subplots(rows, columns, figsize=(5.6 * columns, 4.6 * rows),
                             squeeze=False)
    grid = np.linspace(0.0, 1.0, 400)
    summary = []

    for ax, (label, (spec, frame)) in zip(axes.ravel(), sampled.items()):
        A, B = spec.get("A", 0.1), spec.get("B", 1.0)
        name = spec.get("r_transform", "?")
        # Thin out purely for legibility; the summary table still uses every point.
        shown = frame.sample(max_points, random_state=0) if len(frame) > max_points else frame

        ax.axhspan(A, B, color="0.88", alpha=0.6, zorder=0)
        curve = transform_curve(spec, grid)
        if curve is not None:
            ax.plot(grid, curve, color="0.35", linewidth=1.0, linestyle=":",
                    zorder=2, label="analytic (unclamped)")
            ax.plot(grid, np.clip(curve, A, B), color="black", linewidth=2.0,
                    zorder=3, label="after clamp")
        points = ax.scatter(shown["S_hat"], shown["R"], c=shown["step"], cmap="viridis",
                            s=6, alpha=0.5, edgecolors="none", zorder=4)
        # Marginal S_hat density, drawn as a rug strip just under the top edge.
        density, edges = np.histogram(frame["S_hat"], bins=60, range=(0, 1), density=True)
        if density.max() > 0:
            centers = 0.5 * (edges[:-1] + edges[1:])
            top, height = B + 0.06 * (B - A), 0.10 * (B - A)
            ax.fill_between(centers, top, top + height * density / density.max(),
                            color=TRANSFORM_COLORS.get(name, "#7f7f7f"), alpha=0.55,
                            linewidth=0, zorder=4)

        at_low = float((frame["R"] <= A + 1e-6).mean())
        at_high = float((frame["R"] >= B - 1e-6).mean())
        summary.append({
            "transform": label, "samples": len(frame),
            "S_hat_p05": frame["S_hat"].quantile(0.05),
            "S_hat_median": frame["S_hat"].median(),
            "S_hat_p95": frame["S_hat"].quantile(0.95),
            "R_median": frame["R"].median(),
            "R_span": frame["R"].max() - frame["R"].min(),
            "band_used_%": 100 * (frame["R"].max() - frame["R"].min()) / (B - A),
            "clamped_low_%": 100 * at_low,
            "clamped_high_%": 100 * at_high,
        })

        # Fixed limits rather than autoscaled ones, so panels stay comparable and
        # the rug strip above B always has room.
        floor = min(A, curve.min()) if curve is not None else A
        ax.set(xlabel="$\\hat{S}_t$", ylabel="$R_t$", xlim=(0, 1),
               ylim=(floor - 0.05 * (B - A), B + 0.20 * (B - A)),
               title=f"{label}\nmedian $\\hat{{S}}$={frame['S_hat'].median():.3f}, "
                     f"$R$ span={frame['R'].max() - frame['R'].min():.3f}")
        if curve is not None:
            ax.legend(fontsize=8, loc="lower left")
        else:
            ax.text(0.97, 0.55, "zscore: state-dependent,\nno fixed curve",
                    transform=ax.transAxes, fontsize=8.5, ha="right",
                    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85))
        fig.colorbar(points, ax=ax, label="optimizer step", pad=0.02)

    for spare in axes.ravel()[len(sampled):]:
        spare.axis("off")
    fig.suptitle(f"{task} -- R-transform operating points over their analytic curves",
                 fontsize=12, y=1.01)
    save_fig(fig, "transform_operating_points", task)
    plt.show()

    table = pd.DataFrame(summary).set_index("transform")
    print("'band_used_%' is the fraction of the [A, B] clamp range the transform "
          "actually exercised -- a low value means R was nearly constant, i.e. the "
          "modulation was inert and the optimizer reduced to plain momentum SGD.")
    display(table.round(3))
    return table


transform_table = plot_transform_operating_points(TASK)

### 7.3 Which transform is worth using?

`sweep_r_transforms.py` saves each transform as its own optimizer (`ema_exp`,
`ema_sigmoid`, ...), so if you ran the sweep on this task they are already
competing in **every** section of this notebook -- the leaderboard in section 3,
the learning curves in section 4, the variance comparison in section 5, and the
significance test in section 9 all rank them against each other with no extra
work.

The cell below joins the two halves of the question: how much of its clamp range
each transform actually used (section 7.2) against what that bought in final
performance (section 3).

A transform with a wide `band_used_%` and a poor score is modulating
aggressively in the wrong direction; one with a near-zero `band_used_%` is not
really modulating at all, and any score it posts is attributable to its
effective learning rate, not to DeltaGrad's mechanism.

In [ ]:
def compare_transform_outcomes(task, table):
    if table is None or table.empty:
        return _note("No transform samples to join against the leaderboard.")

    scores, _ = leaderboard(task, load_task(task), reference_optimizer(load_task(task)))
    joined = table.join(scores[["final", "final_std", "grad_var"]], how="left")
    joined = joined.sort_values("final", ascending=lower_is_better(task))

    if joined["final"].isna().all():
        return _note("Sampled runs are not in this task's leaderboard.")

    fig, ax = plt.subplots(figsize=(8.0, 5.0))
    for label, row in joined.iterrows():
        if pd.isna(row["final"]):
            continue
        color = style(label)[0]
        ax.scatter(row["band_used_%"], row["final"], s=140, color=color,
                   edgecolors="black", linewidths=0.8, zorder=3)
        ax.annotate(label.replace("ema_", ""), (row["band_used_%"], row["final"]),
                    textcoords="offset points", xytext=(8, 4), fontsize=9)
        if not pd.isna(row["final_std"]):
            ax.errorbar(row["band_used_%"], row["final"], yerr=row["final_std"],
                        color=color, capsize=3, linewidth=1.2, zorder=2)
    ax.set(xlabel="% of the [A, B] clamp range actually exercised",
           ylabel=metric_name(task),
           title=f"{task} -- does modulating more actually help?")
    save_fig(fig, "transform_band_vs_score", task)
    plt.show()

    display(joined[["samples", "S_hat_median", "R_median", "band_used_%",
                    "clamped_low_%", "clamped_high_%", "final", "final_std",
                    "grad_var"]].round(3))


compare_transform_outcomes(TASK, transform_table)

## 8. Seed stability

Sec. 4.1 of the plan measures "validation accuracy standard deviation" across
independent runs -- robustness is a headline claim, not a footnote, so it gets
its own section rather than a $\pm$ in a table.

- **Left:** the spread of final-epoch results across seeds. Each dot is one seed;
  the box is the quartiles. A tall box means the reported mean is fragile.
- **Right:** across-seed std at *every* epoch. A curve that stays low means the
  optimizer is reproducible throughout training, not just lucky at the end.

In [ ]:
def plot_seed_stability(task, runs_by_optimizer):
    if not runs_by_optimizer:
        return _note("No results loaded.")

    finals = pd.DataFrame([
        {"optimizer": optimizer, "final": value, "seed_index": i + 1}
        for optimizer, result in runs_by_optimizer.items()
        for i, value in enumerate(final_metric(result))
    ])
    if finals.empty:
        return _note("No final-epoch metrics found.")

    order = list(runs_by_optimizer)
    palette = {o: style(o)[0] for o in order}

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
    sns.boxplot(data=finals, x="optimizer", y="final", order=order, hue="optimizer",
                palette=palette, legend=False, width=0.55, showfliers=False,
                ax=axes[0], boxprops={"alpha": 0.45})
    sns.stripplot(data=finals, x="optimizer", y="final", order=order, hue="optimizer",
                  palette=palette, legend=False, size=7, edgecolor="black",
                  linewidth=0.6, jitter=0.12, ax=axes[0])
    axes[0].set(xlabel="", ylabel=metric_name(task),
                title=f"{task} -- final-epoch spread across seeds")
    axes[0].tick_params(axis="x", rotation=30)

    for optimizer, result in runs_by_optimizer.items():
        matrix = as_matrix(result, "acc_history")
        if matrix.shape[0] < 2:
            continue
        color, dashes = style(optimizer)
        axes[1].plot(np.arange(1, matrix.shape[1] + 1), matrix.std(axis=0, ddof=1),
                     color=color, linestyle=dashes, linewidth=1.6, label=optimizer)
    axes[1].set(xlabel="Epoch", ylabel="Std across seeds",
                title="Across-seed std per epoch (lower = more reproducible)")
    axes[1].legend(fontsize=9, ncol=2)

    save_fig(fig, "seed_stability", task)
    plt.show()

    stability = (finals.groupby("optimizer")["final"]
                 .agg(mean="mean", std="std", spread=lambda s: s.max() - s.min())
                 .reindex(order))
    print("Final-epoch metric across seeds:")
    display(stability.round(4))


plot_seed_stability(TASK, runs)

## 9. Is the difference real?

With 5 seeds per optimizer, a 1-point accuracy gap can easily be noise. Every
comparison below is against the reference optimizer, on final-epoch metrics:

- **Welch's t-test** -- does not assume equal variances between the two
  optimizers, which matters here precisely because differing variance is one of
  the things being claimed.
- **Mann-Whitney U** -- rank-based, so a single outlier seed cannot manufacture
  significance on its own.
- **Cohen's d** -- effect size. With n=5 the p-value is underpowered; d says
  whether the gap is *large*, independently of whether it cleared 0.05.

The forest plot shows each optimizer's mean difference from the reference with a
95% Welch confidence interval. **An interval crossing the dashed zero line means
the difference is not statistically distinguishable from noise.**

> With 5 runs these tests are indicative, not conclusive. Treat a bare
> `p < 0.05` here as "worth running more seeds", not as a result.

In [ ]:
def compare_to_reference(task, runs_by_optimizer, reference=None):
    reference = reference or reference_optimizer(runs_by_optimizer)
    if reference not in runs_by_optimizer or len(runs_by_optimizer) < 2:
        return _note("Need at least two optimizers on this task to compare.")

    baseline = final_metric(runs_by_optimizer[reference])
    if len(baseline) < 2:
        return _note("Need at least 2 runs per optimizer for a variance estimate.")

    lower = lower_is_better(task)
    rows = []
    for optimizer, result in runs_by_optimizer.items():
        if optimizer == reference:
            continue
        values = final_metric(result)
        if len(values) < 2:
            continue
        difference = values.mean() - baseline.mean()
        t_stat, t_p = stats.ttest_ind(values, baseline, equal_var=False)
        u_p = stats.mannwhitneyu(values, baseline, alternative="two-sided")[1]
        pooled = np.sqrt(((len(values) - 1) * values.var(ddof=1)
                          + (len(baseline) - 1) * baseline.var(ddof=1))
                         / (len(values) + len(baseline) - 2))
        # Welch-Satterthwaite dof, so the CI matches the t-test above it.
        se = np.sqrt(values.var(ddof=1) / len(values) + baseline.var(ddof=1) / len(baseline))
        dof = se ** 4 / ((values.var(ddof=1) / len(values)) ** 2 / (len(values) - 1)
                         + (baseline.var(ddof=1) / len(baseline)) ** 2 / (len(baseline) - 1))
        half_width = stats.t.ppf(0.975, dof) * se
        rows.append({
            "optimizer": optimizer,
            "difference": difference,
            "ci_low": difference - half_width,
            "ci_high": difference + half_width,
            "welch_p": t_p,
            "mannwhitney_p": u_p,
            "cohens_d": difference / pooled if pooled else np.nan,
            "better": (difference < 0) if lower else (difference > 0),
        })

    if not rows:
        return _note("No comparable optimizers (each needs >= 2 runs).")
    table = pd.DataFrame(rows).set_index("optimizer")

    fig, ax = plt.subplots(figsize=(8.5, 0.75 * len(table) + 2.2))
    y = np.arange(len(table))
    for i, (optimizer, row) in enumerate(table.iterrows()):
        color = style(optimizer)[0]
        significant = row["welch_p"] < 0.05
        ax.plot([row["ci_low"], row["ci_high"]], [i, i], color=color,
                linewidth=2.4, alpha=0.85, solid_capstyle="round")
        ax.scatter(row["difference"], i, color=color, s=95, zorder=3,
                   edgecolors="black", linewidths=1.0 if significant else 0.0)
        ax.text(row["ci_high"], i + 0.22,
                f"p={row['welch_p']:.3f}, d={row['cohens_d']:+.2f}", fontsize=8.5, va="bottom")
    ax.axvline(0, color="black", linestyle="--", linewidth=1.2)
    ax.set(yticks=y, yticklabels=table.index,
           xlabel=f"Mean difference in {metric_name(task)} vs {reference}"
                  f"  ({'left' if lower else 'right'} of 0 is better)",
           title=f"{task} -- final-epoch difference vs {reference} (95% Welch CI)")
    ax.set_ylim(-0.6, len(table) - 0.2)
    save_fig(fig, "significance_vs_reference", task)
    plt.show()

    display(table.style.format({
        "difference": "{:+.3f}", "ci_low": "{:+.3f}", "ci_high": "{:+.3f}",
        "welch_p": "{:.4f}", "mannwhitney_p": "{:.4f}", "cohens_d": "{:+.2f}",
    }))
    inconclusive = table[(table.ci_low < 0) & (table.ci_high > 0)].index.tolist()
    if inconclusive:
        print("CI crosses zero (indistinguishable from seed noise): "
              + ", ".join(inconclusive))


compare_to_reference(TASK, runs, REFERENCE)

## 10. Wall-clock overhead

Sec. 4.2 budgets **under 0.5%** per-epoch overhead for the extra state DeltaGrad
carries ($(K{+}1)d$ for windowed, $3d$ for EMA, against Adam's $2d$).

Two caveats on reading this:

1. These timings include the gradient-variance instrumentation, which fires 8
   extra forward/backward passes every 10th batch for *every* optimizer. That
   inflates the denominator and so **understates** the true relative overhead.
   For a clean measurement use `experiments/ablation_wallclock.py`.
2. Epoch 1 usually includes warm-up (dataset caching, CUDA kernel autotuning),
   so the median is quoted alongside the mean.

In [ ]:
def plot_wallclock(task, runs_by_optimizer, reference=None):
    reference = reference or reference_optimizer(runs_by_optimizer)
    usable = {o: r for o, r in runs_by_optimizer.items() if seconds_per_epoch(r).size}
    if not usable:
        return _note("No per-epoch timings in these results.")

    rows = []
    for optimizer, result in usable.items():
        per_epoch = seconds_per_epoch(result)
        steady = per_epoch[:, 1:] if per_epoch.shape[1] > 1 else per_epoch   # drop warm-up epoch
        rows.append({
            "optimizer": optimizer,
            "mean_s": per_epoch.mean(),
            "median_s": np.median(steady),
            "std_s": per_epoch.mean(axis=1).std(ddof=1) if per_epoch.shape[0] > 1 else np.nan,
            "total_s": np.mean(result["all_total_times"]),
        })
    table = pd.DataFrame(rows).set_index("optimizer")
    if reference in table.index:
        table["overhead_%"] = (table["median_s"] / table.loc[reference, "median_s"] - 1) * 100

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
    order = table.sort_values("median_s").index
    colors = [style(o)[0] for o in order]
    axes[0].barh(order, table.loc[order, "median_s"], color=colors, alpha=0.85,
                 xerr=table.loc[order, "std_s"].fillna(0), capsize=3)
    axes[0].set(xlabel="Median seconds per epoch (warm-up epoch excluded)",
                title=f"{task} -- per-epoch cost")

    for optimizer, result in usable.items():
        color, dashes = style(optimizer)
        per_epoch = seconds_per_epoch(result)
        axes[1].plot(np.arange(1, per_epoch.shape[1] + 1), per_epoch.mean(axis=0),
                     color=color, linestyle=dashes, linewidth=1.5, label=optimizer)
    axes[1].set(xlabel="Epoch", ylabel="Seconds", title="Per-epoch time over training")
    axes[1].legend(fontsize=9, ncol=2)
    save_fig(fig, "wallclock", task)
    plt.show()

    display(table.style.format({"mean_s": "{:.2f}", "median_s": "{:.2f}",
                                "std_s": "{:.2f}", "total_s": "{:.1f}",
                                "overhead_%": "{:+.2f}%"}, na_rep="--"))
    if "overhead_%" not in table:
        return print("No baseline to compare against -- run e.g. adam on this task "
                     "to measure DeltaGrad's overhead against it.")
    for optimizer in DELTAGRAD_KEYS:
        if optimizer in table.index:
            overhead = table.loc[optimizer, "overhead_%"]
            verdict = "within" if abs(overhead) < 0.5 else "OVER"
            print(f"{optimizer}: {overhead:+.2f}% vs {reference} -- {verdict} "
                  f"the plan's 0.5% budget (instrumentation included; "
                  f"see ablation_wallclock.py for a clean number)")


plot_wallclock(TASK, runs, REFERENCE)

## 11. Noise memorization

The plan's motivating claim (Sec. 1): DeltaGrad "mitigates noise memorization".
`cifar100_noise_0` and `cifar100_noise_20` are the same model and schedule
differing only in 20% randomly flipped training labels, which makes them a
controlled test of exactly that.

Memorization shows up two ways, and this section measures both:

- **Accuracy drop** (clean $\rightarrow$ noisy): how much test accuracy the corrupted
  labels cost. Smaller is more robust.
- **Peak-to-final decay**: within the noisy run, how far test accuracy slid back
  from its own peak. This is memorization happening in real time -- the model
  first learns the signal, then starts fitting the flipped labels, and test
  accuracy falls while training loss keeps improving.

Needs both tasks run. If one is missing the cell says so.

In [ ]:
def compare_noise(clean_task="cifar100_noise_0", noisy_task="cifar100_noise_20"):
    available = set(discover()["task"]) if not discover().empty else set()
    missing = [t for t in (clean_task, noisy_task) if t not in available]
    if missing:
        return _note("Needs both tasks; missing " + ", ".join(missing) + ". Run e.g. "
                     f"`python -m experiments.run_task --task {missing[0]} --optimizer adam`")

    clean, noisy = load_task(clean_task), load_task(noisy_task)
    shared = ordered_optimizers(set(clean) & set(noisy))
    if not shared:
        return _note("No optimizer has results on both the clean and noisy task.")

    rows = []
    for optimizer in shared:
        clean_final = final_metric(clean[optimizer])
        noisy_final = final_metric(noisy[optimizer])
        rows.append({
            "optimizer": optimizer,
            "clean_final": clean_final.mean(),
            "noisy_final": noisy_final.mean(),
            "accuracy_drop": clean_final.mean() - noisy_final.mean(),
            "noisy_peak": best_metric(noisy[optimizer], noisy_task).mean(),
            "peak_to_final_decay": peak_to_final_gap(noisy[optimizer], noisy_task).mean(),
        })
    table = pd.DataFrame(rows).set_index("optimizer")

    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.3))
    for optimizer in shared:
        color, _ = style(optimizer)
        matrix = as_matrix(noisy[optimizer], "acc_history")
        mean = matrix.mean(axis=0)
        epochs = np.arange(1, len(mean) + 1)
        axes[0].plot(epochs, mean, color=color, linewidth=1.7, label=optimizer)
        peak = int(np.argmax(mean))
        axes[0].scatter([peak + 1], [mean[peak]], color=color, s=45, zorder=3,
                        marker="v", edgecolors="black", linewidths=0.6)
    axes[0].set(xlabel="Epoch", ylabel=metric_name(noisy_task),
                title=f"{noisy_task} -- markers show each curve's peak")
    axes[0].legend(fontsize=9)

    x = np.arange(len(shared))
    colors = [style(o)[0] for o in shared]
    axes[1].bar(x - 0.2, table.loc[shared, "clean_final"], 0.4, color=colors, alpha=0.9)
    axes[1].bar(x + 0.2, table.loc[shared, "noisy_final"], 0.4, color=colors,
                alpha=0.45, hatch="//")
    axes[1].set(xticks=x, xticklabels=shared, ylabel=metric_name(clean_task),
                title="Clean vs 20% label noise")
    axes[1].tick_params(axis="x", rotation=30)
    # Proxy handles: the bars are coloured by optimizer, so the legend has to
    # carry the clean/noisy distinction on its own.
    axes[1].legend(handles=[
        mpatches.Patch(facecolor="0.4", alpha=0.9, label=clean_task),
        mpatches.Patch(facecolor="0.4", alpha=0.45, hatch="//", label=noisy_task),
    ], fontsize=9)

    axes[2].bar(shared, table.loc[shared, "peak_to_final_decay"],
                color=[style(o)[0] for o in shared], alpha=0.85)
    axes[2].set(ylabel="Peak accuracy - final accuracy (pp)",
                title="Memorization: accuracy given back after peaking\n(lower = more robust)")
    axes[2].tick_params(axis="x", rotation=30)

    save_fig(fig, "noise_memorization", "_cross_task")
    plt.show()
    display(table.round(3))


compare_noise()

## 12. Cross-task ranking

An optimizer that wins on one task may be a poor default. This ranks every
optimizer *within* each task (1 = best), which is the only comparison that makes
sense when the metric ranges differ so much -- 100-class CIFAR accuracy, binary
IMDB accuracy and VAE reconstruction loss don't share an axis. Rank direction
already accounts for VAE tasks being minimised.

Cells are annotated with the raw metric so absolute performance stays visible;
blank cells are pairs that have not been run.

In [ ]:
def plot_cross_task_ranks():
    if inventory.empty or inventory["task"].nunique() < 2:
        return _note("Needs results on at least two tasks to compare across them.")

    records = []
    for task in sorted(inventory["task"].unique()):
        for optimizer, result in load_task(task).items():
            records.append({"task": task, "optimizer": optimizer,
                            "score": final_metric(result).mean()})
    frame = pd.DataFrame(records)
    scores = frame.pivot(index="task", columns="optimizer", values="score")
    scores = scores.reindex(columns=ordered_optimizers(scores.columns))

    ranks = pd.DataFrame(index=scores.index, columns=scores.columns, dtype=float)
    for task in scores.index:
        # VAE tasks are minimised, so rank direction has to follow the task.
        ranks.loc[task] = scores.loc[task].rank(ascending=lower_is_better(task))

    annotations = pd.DataFrame(
        [["" if pd.isna(ranks.loc[task, optimizer])
          else f"#{int(ranks.loc[task, optimizer])}\n{scores.loc[task, optimizer]:.2f}"
          for optimizer in scores.columns] for task in scores.index],
        index=scores.index, columns=scores.columns)

    fig, ax = plt.subplots(figsize=(1.6 * len(scores.columns) + 3.5,
                                    0.95 * len(scores.index) + 2.4))
    sns.heatmap(ranks.astype(float), annot=annotations.values, fmt="", cmap="RdYlGn_r",
                linewidths=0.6, linecolor="white", ax=ax,
                cbar_kws={"label": "Rank within task (1 = best)"})
    ax.set(xlabel="", ylabel="", title="Final-epoch metric: rank within each task")
    save_fig(fig, "cross_task_ranks", "_cross_task")
    plt.show()

    mean_rank = ranks.mean().sort_values()
    print("Mean rank across tasks (only over tasks where each was actually run):")
    display(mean_rank.to_frame("mean_rank").join(
        ranks.notna().sum().to_frame("tasks_run")).round(2))


plot_cross_task_ranks()

## 13. What was written

Figures land in `results/figures/<task>/` (and `results/figures/_cross_task/` for
sections 11-12), as PNG + PDF. Set `SAVE_FIGURES = False` in section 1.2 to keep
them in-notebook only.

To fill the gaps that made sections above skip themselves:

```bash
# a second optimizer on the same task unlocks sections 9-10
python -m experiments.run_task --task cifar100_noise_20 --optimizer adam

# the clean counterpart unlocks the noise-memorization comparison (section 11)
python -m experiments.run_task --task cifar100_noise_0 --optimizer windowed
python -m experiments.run_task --task cifar100_noise_0 --optimizer adam

# more tasks unlock the cross-task ranking (section 12)
python -m experiments.run_task --task mnist_logreg --optimizer ema

# all 6 EMA R-transforms, with (S_hat, R) sampling on, for section 7 -- and they
# then compete as ema_exp/ema_sigmoid/... in every other section too
python -m experiments.sweep_r_transforms --task mnist_logreg --epochs 15

# and to compare against tuned rather than default hyperparameters:
python -m experiments.tune_hyperparams --task cifar100_noise_20 --optimizer windowed adam
python -m experiments.run_task --task cifar100_noise_20 --optimizer windowed --use-tuned
```

In [ ]:
written = sorted(glob.glob(os.path.join(FIG_ROOT, "*", "*.png")))
print(f"{len(written)} figures under {FIG_ROOT}/:")
for path in written:
    print("  " + path)

print("\nSections that could not run are listed as [skipped] above.")